# 💳 Credit Scoring Model

CodeAlpha Machine Learning Internship — Task 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv('../dataset/german_credit.csv')
print(df.shape)
df.head()

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isna().sum().sum())
print('\nTarget distribution:\n', df['credit_risk'].value_counts())

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='credit_risk')
plt.title('Good vs Bad Credit')
plt.xlabel('Credit Risk (0=Good, 1=Bad)')
plt.show()

In [ ]:
X = df.drop(columns=['credit_risk'])
y = df['credit_risk']
cat = X.select_dtypes(include=['object','category']).columns.tolist()
num = X.select_dtypes(include=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
print('Categorical:', len(cat), 'Numerical:', len(num))

In [ ]:
models = {
 'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42)
}
rows=[]
for name, est in models.items():
    pipe=Pipeline([('preprocessor',pre),('model',est)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    prob=pipe.predict_proba(X_test)[:,1]
    rows.append([name, accuracy_score(y_test,pred), precision_score(y_test,pred), recall_score(y_test,pred), f1_score(y_test,pred), roc_auc_score(y_test,prob)])
comparison=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','ROC-AUC'])
comparison

In [ ]:
grid=GridSearchCV(Pipeline([('preprocessor',pre),('model',LogisticRegression(max_iter=3000,random_state=42))]), {'model__C':[0.01,0.1,1,10]}, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train,y_train)
final_model=grid.best_estimator_
pred=final_model.predict(X_test)
prob=final_model.predict_proba(X_test)[:,1]
metrics={
'Accuracy':accuracy_score(y_test,pred),
'Precision':precision_score(y_test,pred),
'Recall':recall_score(y_test,pred),
'F1-Score':f1_score(y_test,pred),
'ROC-AUC':roc_auc_score(y_test,prob),
'Best C':grid.best_params_['model__C']}
metrics

In [ ]:
cm=confusion_matrix(y_test,pred)
ConfusionMatrixDisplay(cm, display_labels=['Good','Bad']).plot()
plt.title('Final Model Confusion Matrix')
plt.show()

## 🌐 Live Dashboard

Run from the project root with `py -m streamlit run app.py` or double-click `START_Credit_Scoring_Dashboard.bat` on Windows.